# Module 9 • Machine Translation

# Lesson 51 • Statistical Machine Translation

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate to Advanced  
**Execution target:** CPU only

---

## Scope

This lesson introduces the foundations of Statistical Machine Translation (SMT)
before moving to neural machine translation in later lessons.

The notebook covers:

- noisy-channel formulation;
- translation models;
- language models;
- word alignment;
- IBM Model 1 intuition;
- lexical translation probabilities;
- phrase-based SMT;
- distortion and reordering;
- decoding;
- beam search intuition;
- BLEU foundations;
- limitations that motivated NMT.

The executable core builds a small offline bilingual corpus and estimates
word-level translation probabilities with an IBM Model 1 style EM procedure.

## Learning Objectives

After completing this lesson, the learner should be able to:

- explain the noisy-channel formulation of SMT;
- distinguish translation-model and language-model probabilities;
- describe word alignment;
- explain IBM Model 1;
- implement EM-based lexical translation estimation;
- inspect learned bilingual probabilities;
- perform simple word-by-word decoding;
- explain phrase-based SMT;
- describe distortion and reordering;
- understand SMT decoding as a search problem;
- explain BLEU at a foundational level;
- identify major weaknesses of SMT.

## Table of Contents

1. Machine Translation Before Neural Networks
2. Statistical Machine Translation
3. Noisy-Channel Model
4. Translation Probability
5. Language Model Probability
6. Bayes Rule Interpretation
7. Parallel Corpora
8. Sentence Alignment
9. Word Alignment
10. IBM Models
11. IBM Model 1
12. Expectation-Maximization
13. Offline Parallel Corpus
14. Vocabulary Construction
15. Initialization
16. E-Step
17. M-Step
18. EM Training
19. Learned Lexical Probabilities
20. Word Alignment Extraction
21. Word-by-Word Decoding
22. Language Modeling
23. Unigram Language Model
24. Bigram Language Model
25. Combining Translation and Fluency
26. Phrase-Based SMT
27. Phrase Extraction
28. Distortion and Reordering
29. SMT Decoding
30. Beam Search Intuition
31. Unknown Words
32. Morphology and SMT
33. Arabic-Specific Challenges
34. Evaluation
35. BLEU Foundations
36. Brevity Penalty
37. Corpus-Level Evaluation
38. Error Analysis
39. Strengths of SMT
40. Limitations of SMT
41. Why NMT Replaced SMT
42. Reproducibility
43. Knowledge Check
44. Exercises
45. Summary and Next Lesson

# 1. Machine Translation Before Neural Networks

Before neural machine translation became dominant, statistical systems represented
translation using probabilities learned from bilingual corpora.

Rather than explicitly writing every grammar rule, SMT estimates how likely one
sentence is to be a translation of another.

In [ ]:
import math
import platform
import random
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)

approaches = pd.DataFrame(
    [
        ("Rule-based MT", "hand-crafted linguistic rules"),
        ("Statistical MT", "probabilistic models from parallel corpora"),
        ("Neural MT", "end-to-end neural sequence modeling"),
    ],
    columns=["Approach", "Core idea"],
)

approaches

# 2. Statistical Machine Translation

SMT models translation as probabilistic inference.

The system searches for the target sentence that maximizes a score derived from
learned statistical models.

# 3. Noisy-Channel Model

The classical formulation is:

\[
\hat{e}
=
rg\max_e P(e)P(f|e)
\]

where:

- \(f\) is the source sentence;
- \(e\) is a candidate target sentence;
- \(P(e)\) is the language-model probability;
- \(P(f|e)\) is the translation-model probability.

# 4. Translation Probability

The translation model estimates how likely source words, phrases, or sentences are
given target-side units.

# 5. Language Model Probability

The language model estimates fluency in the target language.

A strong translation model alone can produce awkward word order. A language model
rewards natural target-language sequences.

# 6. Bayes Rule Interpretation

The noisy-channel formulation follows from:

\[
P(e|f)
=
\frac{P(f|e)P(e)}{P(f)}
\]

Since \(P(f)\) is constant for all candidate translations, it can be omitted during
decoding.

# 7. Parallel Corpora

SMT requires aligned bilingual data.

Example:

```text
English:  the house is small
French:   la maison est petite
```

# 8. Sentence Alignment

Before word alignment, documents must usually be aligned at sentence level.

Sentence alignment can use:

- position;
- length;
- punctuation;
- lexical anchors.

# 9. Word Alignment

Word alignment links source words with their target translations.

Example:

```text
house  <-> maison
small  <-> petite
```

# 10. IBM Models

IBM Models 1–5 are classical probabilistic word-alignment models.

IBM Model 1 is the simplest and assumes every source position is equally likely to
align to every target position.

# 11. IBM Model 1

IBM Model 1 learns lexical translation probabilities:

\[
t(f|e)
\]

meaning the probability of source word \(f\) given target word \(e\).

# 12. Expectation-Maximization

Word alignments are latent variables. IBM Model 1 can therefore be trained with EM:

- **E-step:** estimate fractional alignment counts;
- **M-step:** normalize counts into translation probabilities.

# 13. Offline Parallel Corpus

In [ ]:
parallel_corpus = [
    ("the house", "la maison"),
    ("the book", "le livre"),
    ("a house", "une maison"),
    ("a book", "un livre"),
    ("the small house", "la petite maison"),
    ("the small book", "le petit livre"),
    ("the big house", "la grande maison"),
    ("the big book", "le grand livre"),
]

corpus_frame = pd.DataFrame(
    parallel_corpus,
    columns=["English", "French"],
)

corpus_frame

# 14. Vocabulary Construction

In [ ]:
english_sentences = [
    sentence.split()
    for sentence, _
    in parallel_corpus
]

french_sentences = [
    sentence.split()
    for _, sentence
    in parallel_corpus
]

english_vocab = sorted({
    token
    for sentence in english_sentences
    for token in sentence
})

french_vocab = sorted({
    token
    for sentence in french_sentences
    for token in sentence
})

print("English vocabulary:", english_vocab)
print("French vocabulary:", french_vocab)

# 15. Initialization

Translation probabilities are initialized uniformly.

In [ ]:
def initialize_translation_table(
    target_vocab,
    source_vocab,
):
    initial_probability = (
        1.0
        / len(source_vocab)
    )

    return {
        target_word: {
            source_word: initial_probability
            for source_word
            in source_vocab
        }
        for target_word
        in target_vocab
    }


translation_table = (
    initialize_translation_table(
        english_vocab,
        french_vocab,
    )
)

# 16. E-Step

For each source word, distribute fractional alignment counts across target words.

In [ ]:
def expectation_step(
    target_sentences,
    source_sentences,
    table,
):
    counts = defaultdict(
        lambda: defaultdict(float)
    )

    totals = defaultdict(float)

    for target_sentence, source_sentence in zip(
        target_sentences,
        source_sentences,
    ):
        for source_word in source_sentence:
            denominator = sum(
                table[target_word][source_word]
                for target_word
                in target_sentence
            )

            if denominator == 0:
                continue

            for target_word in target_sentence:
                contribution = (
                    table[target_word][source_word]
                    / denominator
                )

                counts[target_word][source_word] += (
                    contribution
                )

                totals[target_word] += (
                    contribution
                )

    return counts, totals

# 17. M-Step

Normalize expected counts to obtain updated lexical probabilities.

In [ ]:
def maximization_step(
    table,
    counts,
    totals,
):
    updated = {}

    for target_word in table:
        updated[target_word] = {}

        denominator = totals[
            target_word
        ]

        for source_word in table[
            target_word
        ]:
            if denominator == 0:
                updated[target_word][source_word] = (
                    table[target_word][source_word]
                )
            else:
                updated[target_word][source_word] = (
                    counts[target_word][source_word]
                    / denominator
                )

    return updated

# 18. EM Training

In [ ]:
def train_ibm_model_1(
    target_sentences,
    source_sentences,
    iterations=20,
):
    target_vocab = sorted({
        token
        for sentence in target_sentences
        for token in sentence
    })

    source_vocab = sorted({
        token
        for sentence in source_sentences
        for token in sentence
    })

    table = initialize_translation_table(
        target_vocab,
        source_vocab,
    )

    history = []

    for iteration in range(
        iterations
    ):
        counts, totals = expectation_step(
            target_sentences,
            source_sentences,
            table,
        )

        table = maximization_step(
            table,
            counts,
            totals,
        )

        sharpness = np.mean([
            max(
                table[target_word].values()
            )
            for target_word
            in target_vocab
        ])

        history.append({
            "iteration": iteration + 1,
            "mean_max_probability": (
                sharpness
            ),
        })

    return table, pd.DataFrame(history)


translation_table, training_history = (
    train_ibm_model_1(
        english_sentences,
        french_sentences,
        iterations=25,
    )
)

training_history.tail()

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(
    training_history["iteration"],
    training_history[
        "mean_max_probability"
    ],
    marker="o",
)
plt.xlabel("EM iteration")
plt.ylabel("Mean maximum lexical probability")
plt.title("IBM Model 1 Training")
plt.tight_layout()
plt.show()

# 19. Learned Lexical Probabilities

In [ ]:
def top_translations(
    target_word,
    table,
    top_n=5,
):
    items = sorted(
        table[target_word].items(),
        key=lambda item: item[1],
        reverse=True,
    )

    return pd.DataFrame(
        items[:top_n],
        columns=[
            "French word",
            "P(French|English)",
        ],
    )


top_translations(
    "house",
    translation_table,
)

# 20. Word Alignment Extraction

A simple alignment picks the target word with the largest lexical probability for
each source word.

In [ ]:
def best_alignment(
    english_sentence,
    french_sentence,
    table,
):
    alignments = []

    for french_word in french_sentence:
        best_english = max(
            english_sentence,
            key=lambda english_word: (
                table[
                    english_word
                ][
                    french_word
                ]
            ),
        )

        alignments.append(
            (
                best_english,
                french_word,
                table[
                    best_english
                ][
                    french_word
                ],
            )
        )

    return pd.DataFrame(
        alignments,
        columns=[
            "English",
            "French",
            "Probability",
        ],
    )


best_alignment(
    ["the", "small", "house"],
    ["la", "petite", "maison"],
    translation_table,
)

# 21. Word-by-Word Decoding

In [ ]:
inverse_table = defaultdict(dict)

for english_word, french_probs in (
    translation_table.items()
):
    for french_word, probability in (
        french_probs.items()
    ):
        inverse_table[
            french_word
        ][
            english_word
        ] = probability


def word_by_word_translate(
    french_sentence,
):
    output = []

    for french_word in (
        french_sentence.split()
    ):
        candidates = inverse_table[
            french_word
        ]

        if not candidates:
            output.append(
                french_word
            )
            continue

        best = max(
            candidates,
            key=candidates.get,
        )

        output.append(best)

    return " ".join(output)


word_by_word_translate(
    "la petite maison"
)

# 22. Language Modeling

Lexical translation alone does not guarantee fluent output.

A target-language model can score candidate word sequences.

# 23. Unigram Language Model

In [ ]:
english_token_counts = Counter(
    token
    for sentence in english_sentences
    for token in sentence
)

total_english_tokens = sum(
    english_token_counts.values()
)


def unigram_probability(
    word,
    alpha=1.0,
):
    vocabulary_size = len(
        english_vocab
    )

    return (
        english_token_counts[
            word
        ]
        + alpha
    ) / (
        total_english_tokens
        + alpha
        * vocabulary_size
    )

# 24. Bigram Language Model

In [ ]:
bigram_counts = Counter()
context_counts = Counter()

for sentence in english_sentences:
    tokens = [
        "<s>",
        *sentence,
        "</s>",
    ]

    for left, right in zip(
        tokens,
        tokens[1:],
    ):
        bigram_counts[
            (left, right)
        ] += 1

        context_counts[
            left
        ] += 1


def bigram_probability(
    left,
    right,
    alpha=1.0,
):
    vocabulary_size = (
        len(english_vocab)
        + 1
    )

    return (
        bigram_counts[
            (left, right)
        ]
        + alpha
    ) / (
        context_counts[left]
        + alpha
        * vocabulary_size
    )


def sentence_log_probability(
    sentence,
):
    tokens = [
        "<s>",
        *sentence.split(),
        "</s>",
    ]

    return sum(
        math.log(
            bigram_probability(
                left,
                right,
            )
        )
        for left, right
        in zip(
            tokens,
            tokens[1:],
        )
    )


pd.Series({
    "the small house": (
        sentence_log_probability(
            "the small house"
        )
    ),
    "small the house": (
        sentence_log_probability(
            "small the house"
        )
    ),
})

# 25. Combining Translation and Fluency

SMT decoders combine several features, commonly using log-linear models.

A simplified score can combine:

- lexical translation probability;
- language-model probability;
- distortion penalty;
- phrase penalty.

# 26. Phrase-Based SMT

Phrase-based SMT translates multiword segments rather than isolated words.

A phrase is simply a contiguous sequence of tokens; it does not need to be a
linguistic phrase.

# 27. Phrase Extraction

Phrase pairs are typically extracted from word-aligned sentence pairs.

Example:

```text
the house  <-> la maison
small house <-> petite maison
```

In [ ]:
illustrative_phrase_table = pd.DataFrame(
    [
        ("the house", "la maison", 0.88),
        ("a house", "une maison", 0.84),
        ("small house", "petite maison", 0.79),
        ("big house", "grande maison", 0.81),
    ],
    columns=[
        "English phrase",
        "French phrase",
        "Illustrative probability",
    ],
)

illustrative_phrase_table

# 28. Distortion and Reordering

Languages often use different word orders.

Phrase-based systems therefore include distortion or reordering models.

# 29. SMT Decoding

Decoding is a search problem over many possible translations.

Exhaustive search is usually impossible, so approximate search is used.

# 30. Beam Search Intuition

Beam search keeps only the best partial hypotheses at each stage.

In [ ]:
beam_example = pd.DataFrame(
    [
        (1, "the", -0.5),
        (1, "a", -0.7),
        (2, "the house", -0.8),
        (2, "a house", -1.1),
        (2, "house the", -2.2),
    ],
    columns=[
        "Step",
        "Hypothesis",
        "Score",
    ],
)

beam_example

# 31. Unknown Words

SMT systems can struggle with words absent from the phrase table.

Common strategies included:

- copying;
- transliteration;
- dictionaries;
- morphological preprocessing.

# 32. Morphology and SMT

Morphologically rich languages create data sparsity because many surface forms can
correspond to one lemma or stem.

# 33. Arabic-Specific Challenges

Arabic presents several SMT challenges:

- rich inflection and derivation;
- attached clitics;
- optional tashkeel;
- orthographic variation;
- relatively flexible word order.

Preprocessing choices strongly affect alignment quality.

In [ ]:
arabic_examples = pd.DataFrame(
    [
        (
            "وَسَيَكْتُبُونَهَا",
            "fully vocalized complex verb",
        ),
        (
            "بِالْمَدْرَسَةِ",
            "preposition + noun",
        ),
        (
            "كِتَابُهُمَا",
            "noun + dual possessive pronoun",
        ),
    ],
    columns=[
        "Arabic form",
        "Morphological property",
    ],
)

arabic_examples

In a fully vocalized Arabic task, stripping tashkeel would collapse distinctions and
change the experimental setup.

# 34. Evaluation

SMT systems historically relied heavily on automatic corpus-level metrics.

# 35. BLEU Foundations

BLEU compares system output with reference translations using modified n-gram
precision and a brevity penalty.

In [ ]:
def ngrams(
    tokens,
    n,
):
    return [
        tuple(
            tokens[
                index:
                index + n
            ]
        )
        for index in range(
            len(tokens)
            - n
            + 1
        )
    ]


def modified_precision(
    reference,
    candidate,
    n,
):
    reference_counts = Counter(
        ngrams(
            reference.split(),
            n,
        )
    )

    candidate_counts = Counter(
        ngrams(
            candidate.split(),
            n,
        )
    )

    clipped = sum(
        min(
            count,
            reference_counts[
                gram
            ],
        )
        for gram, count
        in candidate_counts.items()
    )

    total = sum(
        candidate_counts.values()
    )

    if total == 0:
        return 0.0

    return clipped / total


modified_precision(
    "the small house",
    "the small house",
    2,
)

# 36. Brevity Penalty

In [ ]:
def brevity_penalty(
    reference_length,
    candidate_length,
):
    if candidate_length == 0:
        return 0.0

    if candidate_length > reference_length:
        return 1.0

    return math.exp(
        1.0
        - reference_length
        / candidate_length
    )


brevity_penalty(
    5,
    4,
)

# 37. Corpus-Level Evaluation

BLEU is most meaningful when computed across a corpus rather than one isolated
sentence.

# 38. Error Analysis

SMT errors can include:

- lexical mistranslation;
- missing words;
- repeated phrases;
- wrong reordering;
- agreement errors;
- unknown-word copying.

In [ ]:
smt_errors = pd.DataFrame(
    [
        ("Lexical", "wrong word translation"),
        ("Coverage", "source word untranslated"),
        ("Reordering", "incorrect phrase order"),
        ("Agreement", "target morphology inconsistent"),
        ("Unknown word", "unseen item copied or dropped"),
    ],
    columns=[
        "Error type",
        "Description",
    ],
)

smt_errors

# 39. Strengths of SMT

SMT offered:

- interpretable phrase tables;
- explicit alignment;
- modular architecture;
- useful probabilistic foundations;
- strong performance before neural MT.

# 40. Limitations of SMT

Major limitations include:

- data sparsity;
- brittle phrase segmentation;
- weak long-distance context;
- separate modules optimized imperfectly together;
- difficult morphology;
- search complexity.

# 41. Why NMT Replaced SMT

Neural MT learns continuous representations and trains an end-to-end model that
jointly captures lexical, syntactic, and contextual dependencies.

Transformer-based NMT further improved long-range modeling and parallel training.

# 42. Reproducibility

In [ ]:
reproducibility = pd.Series(
    {
        "module": "Module 9 • Machine Translation",
        "lesson": "Lesson 51 • Statistical Machine Translation",
        "sentence_pairs": len(parallel_corpus),
        "EM_iterations": 25,
        "english_vocab": len(english_vocab),
        "french_vocab": len(french_vocab),
        "seed": 42,
        "offline_execution": True,
        "python": platform.python_version(),
    },
    name="Lesson 51 experiment",
)

reproducibility

# 43. Knowledge Check

1. What is Statistical Machine Translation?
2. What is the noisy-channel formulation?
3. What does the translation model estimate?
4. What does the language model estimate?
5. What is word alignment?
6. What does IBM Model 1 learn?
7. Why is EM required?
8. What happens in the E-step?
9. What happens in the M-step?
10. What is phrase-based SMT?
11. Why is reordering necessary?
12. Why is decoding a search problem?
13. What does BLEU measure?
14. Why is Arabic challenging for SMT?
15. What limitations motivated NMT?

# 44. Exercises

## Exercise 1
Add more sentence pairs to the bilingual corpus.

## Exercise 2
Increase the EM iterations and inspect probability stability.

## Exercise 3
Train the model in the reverse language direction.

## Exercise 4
Add a NULL alignment token.

## Exercise 5
Visualize a word-alignment matrix.

## Exercise 6
Build a simple phrase table.

## Exercise 7
Add a distortion penalty.

## Exercise 8
Compare unigram and bigram language-model rankings.

## Exercise 9
Add Arabic–English sentence pairs with full tashkeel.

## Exercise 10
Compute BLEU-like scores for multiple candidate translations.

## Challenge Exercises

1. Implement IBM Model 1 with NULL alignment.
2. Implement bidirectional alignments and symmetrization.
3. Extract phrase pairs from word alignments.
4. Build a tiny beam-search phrase-based decoder.
5. Compare SMT output with a neural MT model in a later lesson.

# 45. Summary and Next Lesson

In this lesson:

- the noisy-channel formulation of SMT was introduced;
- translation and language models were separated;
- word alignment and IBM Model 1 were explained;
- EM training was implemented;
- lexical translation probabilities were learned from an offline parallel corpus;
- simple word-level decoding was demonstrated;
- target-language unigram and bigram models were introduced;
- phrase-based SMT, reordering, decoding, beam search, unknown words, BLEU, Arabic
  morphology, and SMT limitations were covered.

## Next Lesson

**Lesson 52: Neural Machine Translation with Encoder–Decoder Networks and
Attention** introduces neural sequence-to-sequence translation, encoder–decoder
training, teacher forcing, attention, greedy decoding, beam search, and an offline
CPU NMT experiment.

# References

- Brown, P. et al. foundational work on statistical machine translation.
- Brown, P. et al. *The Mathematics of Statistical Machine Translation:
  Parameter Estimation*.
- Koehn, P. *Statistical Machine Translation*.
- Koehn, P. et al. *Moses: Open Source Toolkit for Statistical Machine
  Translation*.
- Papineni, K. et al. *BLEU: a Method for Automatic Evaluation of Machine
  Translation*.